# CSE374 Machine Learning Project — Credit Card Default (Taiwan)
## Role 1: Data & Preprocessing Lead

**Dataset:** Default of Credit Card Clients (UCI Repository)  
**Source:** https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients  


---

This notebook covers the full data loading, cleaning, preprocessing pipeline, and train/validation/test split strategy as required by Role 1.

## 1. Imports & Setup

Importing core libraries for data handling (`pandas`, `numpy`), preprocessing (`StandardScaler` from sklearn), splitting (`train_test_split`), and class imbalance handling (`SMOTE` from imbalanced-learn).

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

## 2. Data Loading

The dataset is loaded from the original `.xls` file. I use `header=1` because the first row in this specific file contains a redundant title row, and the actual column names are in row 2. The `ID` column is dropped immediately since it is a unique identifier with no predictive value.

I also rename:
- `PAY_0` → `PAY_1` to match the logical naming convention (PAY_1 through PAY_6)
- `default payment next month` → `default` for cleaner referencing throughout the notebook

In [2]:
file_path = '../data/raw_data.xls'
df = pd.read_excel(file_path, header=1)
df = df.drop(columns=['ID'])
df.head()


,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,20000,2,2,1,24,2,2,-1,-1,-2,...,0,0,0,0,689,0,0,0,0,1
1,120000,2,2,2,26,-1,2,0,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,90000,2,2,2,34,0,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,50000,2,2,1,37,0,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,50000,1,2,1,57,-1,0,-1,0,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


In [3]:
df = df.rename(columns={
    'default payment next month': 'default',
    'PAY_0': 'PAY_1'
})
df.head(1)


,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_1,PAY_2,PAY_3,PAY_4,PAY_5,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default
0,20000,2,2,1,24,2,2,-1,-1,-2,...,0,0,0,0,689,0,0,0,0,1


## 3. Data Inspection

Before any cleaning, I inspect the dataset structure: data types, shape, and whether any missing values exist.

The dataset contains **30,000 instances** and **23 features** after dropping ID. The target variable is `default` (binary: 0 = no default, 1 = default).

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 24 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   LIMIT_BAL  30000 non-null  int64
 1   SEX        30000 non-null  int64
 2   EDUCATION  30000 non-null  int64
 3   MARRIAGE   30000 non-null  int64
 4   AGE        30000 non-null  int64
 5   PAY_1      30000 non-null  int64
 6   PAY_2      30000 non-null  int64
 7   PAY_3      30000 non-null  int64
 8   PAY_4      30000 non-null  int64
 9   PAY_5      30000 non-null  int64
 10  PAY_6      30000 non-null  int64
 11  BILL_AMT1  30000 non-null  int64
 12  BILL_AMT2  30000 non-null  int64
 13  BILL_AMT3  30000 non-null  int64
 14  BILL_AMT4  30000 non-null  int64
 15  BILL_AMT5  30000 non-null  int64
 16  BILL_AMT6  30000 non-null  int64
 17  PAY_AMT1   30000 non-null  int64
 18  PAY_AMT2   30000 non-null  int64
 19  PAY_AMT3   30000 non-null  int64
 20  PAY_AMT4   30000 non-null  int64
 21  PAY_AMT5   30000 non-nu

In [5]:
print(df.isnull().sum())

LIMIT_BAL    0
SEX          0
EDUCATION    0
MARRIAGE     0
AGE          0
PAY_1        0
PAY_2        0
PAY_3        0
PAY_4        0
PAY_5        0
PAY_6        0
BILL_AMT1    0
BILL_AMT2    0
BILL_AMT3    0
BILL_AMT4    0
BILL_AMT5    0
BILL_AMT6    0
PAY_AMT1     0
PAY_AMT2     0
PAY_AMT3     0
PAY_AMT4     0
PAY_AMT5     0
PAY_AMT6     0
default      0
dtype: int64


## 4. Descriptive Statistics & Categorical Exploration

I examine the distribution of categorical columns (`EDUCATION`, `MARRIAGE`, `SEX`) and numerical columns separately.

**Key finding:** The `EDUCATION` and `MARRIAGE` columns contain undocumented category values (0, 5, 6 in EDUCATION; 0 in MARRIAGE) that do not appear in the official dataset codebook. These are treated as "other/unknown" and merged into the existing catch-all categories in the next step.

In [6]:
df.describe()

,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_1,PAY_2,PAY_3,PAY_4,PAY_5,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default
count,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,...,30000.000000,30000.000000,30000.000000,30000.000000,3.000000e+04,30000.00000,30000.000000,30000.000000,30000.000000,30000.000000
mean,167484.322667,1.603733,1.853133,1.551867,35.485500,-0.016700,-0.133767,-0.166200,-0.220667,-0.266200,...,43262.948967,40311.400967,38871.760400,5663.580500,5.921163e+03,5225.68150,4826.076867,4799.387633,5215.502567,0.221200
std,129747.661567,0.489129,0.790349,0.521970,9.217904,1.123802,1.197186,1.196868,1.169139,1.133187,...,64332.856134,60797.155770,59554.107537,16563.280354,2.304087e+04,17606.96147,15666.159744,15278.305679,17777.465775,0.415062
min,10000.000000,1.000000,0.000000,0.000000,21.000000,-2.000000,-2.000000,-2.000000,-2.000000,-2.000000,...,-170000.000000,-81334.000000,-339603.000000,0.000000,0.000000e+00,0.00000,0.000000,0.000000,0.000000,0.000000
25%,50000.000000,1.000000,1.000000,1.000000,28.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,...,2326.750000,1763.000000,1256.000000,1000.000000,8.330000e+02,390.00000,296.000000,252.500000,117.750000,0.000000
50%,140000.000000,2.000000,2.000000,2.000000,34.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,19052.000000,18104.500000,17071.000000,2100.000000,2.009000e+03,1800.00000,1500.000000,1500.000000,1500.000000,0.000000
75%,240000.000000,2.000000,2.000000,2.000000,41.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,54506.000000,50190.500000,49198.250000,5006.000000,5.000000e+03,4505.00000,4013.250000,4031.500000,4000.000000,0.000000
max,1000000.000000,2.000000,6.000000,3.000000,79.000000,8.000000,8.000000,8.000000,8.000000,8.000000,...,891586.000000,927171.000000,961664.000000,873552.000000,1.684259e+06,896040.00000,621000.000000,426529.000000,528666.000000,1.000000


In [7]:
print(df['EDUCATION'].value_counts().sort_index())
print(df['MARRIAGE'].value_counts().sort_index())
print(df['SEX'].value_counts().sort_index())

EDUCATION
0       14
1    10585
2    14030
3     4917
4      123
5      280
6       51
Name: count, dtype: int64
MARRIAGE
0       54
1    13659
2    15964
3      323
Name: count, dtype: int64
SEX
1    11888
2    18112
Name: count, dtype: int64


## 5. Data Cleaning — Undocumented Categories

**EDUCATION:** The codebook defines values 1 (graduate school), 2 (university), 3 (high school), 4 (others). Values 0, 5, and 6 are undocumented and likely represent edge cases or data entry errors. I map all three → 4 (others).

**MARRIAGE:** The codebook defines values 1 (married), 2 (single), 3 (others). Value 0 is undocumented. I map 0 → 3 (others).

**No other encoding is needed** for categorical variables (`SEX`, `EDUCATION`, `MARRIAGE`) because they are already numerically encoded in the original dataset.

In [8]:
print(df['EDUCATION'].value_counts().sort_index())
print(df['MARRIAGE'].value_counts().sort_index())
print(df['SEX'].value_counts().sort_index())

EDUCATION
0       14
1    10585
2    14030
3     4917
4      123
5      280
6       51
Name: count, dtype: int64
MARRIAGE
0       54
1    13659
2    15964
3      323
Name: count, dtype: int64
SEX
1    11888
2    18112
Name: count, dtype: int64


In [9]:
df['EDUCATION'] = df['EDUCATION'].replace({0: 4, 5: 4, 6: 4})
df['MARRIAGE'] = df['MARRIAGE'].replace({0: 3})

In [10]:
print(df['EDUCATION'].value_counts().sort_index())
print(df['MARRIAGE'].value_counts().sort_index())
print(df['SEX'].value_counts().sort_index())

EDUCATION
1    10585
2    14030
3     4917
4      468
Name: count, dtype: int64
MARRIAGE
1    13659
2    15964
3      377
Name: count, dtype: int64
SEX
1    11888
2    18112
Name: count, dtype: int64


## 6. Inspection of PAY Status Columns

I inspect the repayment status columns (`PAY_1` through `PAY_6`). According to the dataset documentation:
- `-2` = no consumption that month
- `-1` = paid in full (duly)
- `0` = use of revolving credit
- `1–9` = payment delayed by N months

These negative values are **valid and expected** — they are not errors and will be kept as-is. No cleaning is applied here.

In [11]:
pay_columns = ['PAY_1', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']
for col in pay_columns:
    print(f"{col}:")
    print(df[col].value_counts().sort_index())

PAY_1:
PAY_1
-2     2759
-1     5686
 0    14737
 1     3688
 2     2667
 3      322
 4       76
 5       26
 6       11
 7        9
 8       19
Name: count, dtype: int64
PAY_2:
PAY_2
-2     3782
-1     6050
 0    15730
 1       28
 2     3927
 3      326
 4       99
 5       25
 6       12
 7       20
 8        1
Name: count, dtype: int64
PAY_3:
PAY_3
-2     4085
-1     5938
 0    15764
 1        4
 2     3819
 3      240
 4       76
 5       21
 6       23
 7       27
 8        3
Name: count, dtype: int64
PAY_4:
PAY_4
-2     4348
-1     5687
 0    16455
 1        2
 2     3159
 3      180
 4       69
 5       35
 6        5
 7       58
 8        2
Name: count, dtype: int64
PAY_5:
PAY_5
-2     4546
-1     5539
 0    16947
 2     2626
 3      178
 4       84
 5       17
 6        4
 7       58
 8        1
Name: count, dtype: int64
PAY_6:
PAY_6
-2     4895
-1     5740
 0    16286
 2     2766
 3      184
 4       49
 5       13
 6       19
 7       46
 8        2
Name: count, dtype: int6

In [12]:
df[['LIMIT_BAL', 'AGE']].describe()

,LIMIT_BAL,AGE
count,30000.000000,30000.000000
mean,167484.322667,35.485500
std,129747.661567,9.217904
min,10000.000000,21.000000
25%,50000.000000,28.000000
50%,140000.000000,34.000000
75%,240000.000000,41.000000
max,1000000.000000,79.000000


## 7. Inspection of Bill Amount Columns (BILL_AMT1–6)

I check for negative values in bill statement amounts. **Negative BILL_AMT values are not errors** — they represent a credit balance, meaning the client overpaid in a previous month and has a positive balance on their account. These values are kept as-is since they carry genuine financial signal.

In [13]:
bill_columns = ['BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6']
df[bill_columns].describe()

,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6
count,30000.000000,30000.000000,3.000000e+04,30000.000000,30000.000000,30000.000000
mean,51223.330900,49179.075167,4.701315e+04,43262.948967,40311.400967,38871.760400
std,73635.860576,71173.768783,6.934939e+04,64332.856134,60797.155770,59554.107537
min,-165580.000000,-69777.000000,-1.572640e+05,-170000.000000,-81334.000000,-339603.000000
25%,3558.750000,2984.750000,2.666250e+03,2326.750000,1763.000000,1256.000000
50%,22381.500000,21200.000000,2.008850e+04,19052.000000,18104.500000,17071.000000
75%,67091.000000,64006.250000,6.016475e+04,54506.000000,50190.500000,49198.250000
max,964511.000000,983931.000000,1.664089e+06,891586.000000,927171.000000,961664.000000


In [14]:
for col in bill_columns:
    neg_count = (df[col] < 0).sum()
    print(f"{col}: {neg_count} values")

BILL_AMT1: 590 values
BILL_AMT2: 669 values
BILL_AMT3: 655 values
BILL_AMT4: 675 values
BILL_AMT5: 655 values
BILL_AMT6: 688 values


## 8. Inspection of Payment Amount Columns (PAY_AMT1–6)

I verify that actual payment amounts contain no negative values. Payment amounts by definition cannot be negative (you cannot pay a negative amount). Any negatives here would indicate data corruption. As confirmed by the check below, none exist.

**Note on large values:** The PAY_AMT columns contain very large maximum values (up to ~1.68M). These are not outliers to be removed — clients with high credit limits (LIMIT_BAL up to 1,000,000) naturally make large payments. Removing or capping them would discard real financial behavior. StandardScaler in Section 10 will handle the wide range by normalizing these values without discarding the information they carry.


In [15]:
pay_amt_columns = ['PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']
df[pay_amt_columns].describe()

,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6
count,30000.000000,3.000000e+04,30000.00000,30000.000000,30000.000000,30000.000000
mean,5663.580500,5.921163e+03,5225.68150,4826.076867,4799.387633,5215.502567
std,16563.280354,2.304087e+04,17606.96147,15666.159744,15278.305679,17777.465775
min,0.000000,0.000000e+00,0.00000,0.000000,0.000000,0.000000
25%,1000.000000,8.330000e+02,390.00000,296.000000,252.500000,117.750000
50%,2100.000000,2.009000e+03,1800.00000,1500.000000,1500.000000,1500.000000
75%,5006.000000,5.000000e+03,4505.00000,4013.250000,4031.500000,4000.000000
max,873552.000000,1.684259e+06,896040.00000,621000.000000,426529.000000,528666.000000


In [16]:
for col in pay_amt_columns:
    neg_count = (df[col] < 0).sum()
    if neg_count > 0:
      print(f"{col}: {neg_count} values")
print("✓ No negative values found in any PAY_AMT column.")


✓ No negative values found in any PAY_AMT column.


## 9. Train / Validation / Test Split Strategy

I split the data into three sets:
- **Train (64%):** Used to fit all models
- **Validation (16%):** Used during modeling for hyperparameter tuning and model selection — never used to make final decisions
- **Test (20%):** Held out entirely until final evaluation — touched only once

I use `stratify=y` to preserve the original class ratio (~78% non-default, ~22% default) across all three splits. `random_state=42` ensures reproducibility.

> **Note:** SMOTE for class imbalance will be applied **only on the training set** after splitting, to prevent any synthetic samples from leaking into validation or test sets.

In [17]:
X = df.drop(columns=['default'])
y = df['default']

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.2, random_state=42, stratify=y_temp
)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

Train: (19200, 23), Val: (4800, 23), Test: (6000, 23)


## 10. Feature Scaling

Numerical features (`LIMIT_BAL`, `AGE`, `BILL_AMT1–6`, `PAY_AMT1–6`) are scaled using **StandardScaler** (zero mean, unit variance).

**Why StandardScaler and not MinMaxScaler?**  
This dataset contains outliers in bill and payment amounts (some clients have very high credit limits or unusual payment histories). StandardScaler is more robust to outliers than MinMaxScaler, which compresses everything into [0,1] and gets distorted by extreme values.

**Important:** The scaler is fit **only on `X_train`** and then applied to validation and test sets. Fitting on the full dataset would constitute data leakage.

**Why scale only numerical columns?**  
The PAY status columns (PAY_1–6) are ordinal-coded integers with a natural order and bounded range (-2 to 9). The categorical columns (SEX, EDUCATION, MARRIAGE) are nominal codes. Scaling either group would distort their meaning.

In [18]:
num_columns = [
    'LIMIT_BAL', 'AGE',
    'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6',
    'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6'
]
scaler = StandardScaler()
X_train[num_columns] = scaler.fit_transform(X_train[num_columns])
X_val[num_columns]   = scaler.transform(X_val[num_columns])
X_test[num_columns]  = scaler.transform(X_test[num_columns])

In [19]:
for val, count in y_train.value_counts().items():
    perc = (count / len(y_train)) * 100
    print(f"Class {val}: {count} samples ({perc:.2f}%)")

Class 0: 14953 samples (77.88%)
Class 1: 4247 samples (22.12%)


## 11. Handling Class Imbalance — SMOTE

Checking the class distribution in the training set reveals significant imbalance (~78% class 0, ~22% class 1). Training directly on imbalanced data causes the model to be biased toward the majority class (predicting "no default" for almost everything), which is dangerous in a credit risk context where missing a defaulter is costly.

**Why SMOTE?**  
SMOTE (Synthetic Minority Over-sampling Technique) generates synthetic samples for the minority class by interpolating between existing minority examples, rather than simply duplicating them. This produces a more generalizable balance compared to naive oversampling.



In [20]:
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)
print(f" {X_train_balanced.shape}")
print(y_train_balanced.value_counts(normalize=True) * 100)

 (29906, 23)
default
0    50.0
1    50.0
Name: proportion, dtype: float64


## 12. Preprocessing Summary

| Step | Decision | Justification |
|---|---|---|
| Drop ID | Removed | No predictive value |
| Rename columns | PAY_0→PAY_1, target simplified | Consistency |
| Missing values | None found | No action needed |
| EDUCATION/MARRIAGE | Undocumented values merged into 'other' | Codebook compliance |
| Negative BILL_AMT | Kept as-is | Represent valid credit balances |
| PAY status (-2, -1) | Kept as-is | Documented valid values |
| Split strategy | 64/16/20 train/val/test, stratified | Prevent leakage, preserve class ratio |
| Scaling | StandardScaler on numerical only, fit on train | Robust to outliers, no leakage |
| Class imbalance | SMOTE on train only | Avoid bias toward majority class |

**Final shapes:**
- **X_train_balanced:** (29,906 × 23) after SMOTE — minority class upsampled to match majority
- **X_val:** (4,800 × 23) — untouched, reflects real-world distribution
- **X_test:** (6,000 × 23) — held out, touched only at final evaluation

**Note on Feature Selection:**  
No feature selection was performed at this stage, and this is a deliberate decision. The dataset contains only 23 features after dropping ID, which is a very manageable dimensionality — there is no curse of dimensionality concern here. All features carry domain-relevant financial information: credit limit, demographics, 6 months of repayment status, 6 months of bill statements, and 6 months of payment amounts. Dropping any feature group without evidence would risk losing predictive signal.

Feature importance will instead be assessed **after modeling** — tree-based models (e.g. Decision Tree) will naturally reveal which features contribute most, and this can guide any future dimensionality reduction if needed. Performing feature selection before seeing model results would be premature at this stage.


In [21]:
import pickle

# Save all the final splits into your data folder
with open('../data/processed_splits.pkl', 'wb') as f:
    pickle.dump((X_train_balanced, X_val, X_test, y_train_balanced, y_val, y_test), f)

print("✓ Data successfully exported for modeling!")

✓ Data successfully exported for modeling!
